# Feature Consistency Analysis Across Random Seeds

This notebook analyzes how consistent the learned features are across different random seeds for Sparse Autoencoders trained on MNIST.


In [ ]:
import os
import re
import glob
import json
from collections import defaultdict
from itertools import combinations

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")


# =============================================================================
# Model class definitions (TopK SAE from https://arxiv.org/abs/2406.04093)
# =============================================================================

@torch.no_grad()
def set_decoder_norm_to_unit_norm(W_dec, activation_dim, dict_size):
    D, F = W_dec.shape
    assert D == activation_dim, f"Expected activation_dim={activation_dim}, got {D}"
    assert F == dict_size, f"Expected dict_size={dict_size}, got {F}"
    eps = torch.finfo(W_dec.dtype).eps
    norm = torch.norm(W_dec.data, dim=0, keepdim=True)
    W_dec.data /= norm + eps
    return W_dec.data


class AutoEncoderTopK(nn.Module):
    """
    Top-k autoencoder from https://arxiv.org/abs/2406.04093
    """

    def __init__(self, activation_dim: int, dict_size: int, k: int):
        super().__init__()
        self.activation_dim = activation_dim
        self.dict_size = dict_size

        assert isinstance(k, int) and k > 0, f"k={k} must be a positive integer"
        self.register_buffer("k", torch.tensor(k, dtype=torch.int))
        self.register_buffer("threshold", torch.tensor(-1.0, dtype=torch.float32))

        self.decoder = nn.Linear(dict_size, activation_dim, bias=False)
        self.encoder = nn.Linear(activation_dim, dict_size)
        self.b_dec = nn.Parameter(torch.zeros(activation_dim))

    def encode(self, x: torch.Tensor, return_topk: bool = False, use_threshold: bool = False):
        post_relu_feat_acts_BF = F.relu(self.encoder(x - self.b_dec))

        if use_threshold:
            encoded_acts_BF = post_relu_feat_acts_BF * (post_relu_feat_acts_BF > self.threshold)
            if return_topk:
                post_topk = post_relu_feat_acts_BF.topk(self.k, sorted=False, dim=-1)
                return encoded_acts_BF, post_topk.values, post_topk.indices, post_relu_feat_acts_BF
            else:
                return encoded_acts_BF

        post_topk = post_relu_feat_acts_BF.topk(self.k, sorted=False, dim=-1)
        tops_acts_BK = post_topk.values
        top_indices_BK = post_topk.indices

        buffer_BF = torch.zeros_like(post_relu_feat_acts_BF)
        encoded_acts_BF = buffer_BF.scatter_(dim=-1, index=top_indices_BK, src=tops_acts_BK)

        if return_topk:
            return encoded_acts_BF, tops_acts_BK, top_indices_BK, post_relu_feat_acts_BF
        else:
            return encoded_acts_BF

    def decode(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(x) + self.b_dec

    def forward(self, x: torch.Tensor, output_features: bool = False):
        encoded_acts_BF = self.encode(x)
        x_hat_BD = self.decode(encoded_acts_BF)
        if not output_features:
            return x_hat_BD
        else:
            return x_hat_BD, encoded_acts_BF

    @staticmethod
    def from_pretrained(path, k=None, device=None):
        """Load a pretrained autoencoder from a file."""
        state_dict = torch.load(path, map_location='cpu')
        dict_size, activation_dim = state_dict["encoder.weight"].shape

        if k is None:
            k = state_dict["k"].item()
        elif "k" in state_dict and k != state_dict["k"].item():
            raise ValueError(f"k={k} != {state_dict['k'].item()}=state_dict['k']")

        autoencoder = AutoEncoderTopK(activation_dim, dict_size, k)
        autoencoder.load_state_dict(state_dict)
        if device is not None:
            autoencoder.to(device)
        return autoencoder


## 1. Load Models and Organize by Configuration


In [ ]:
def load_config(config_path):
    """
    Load and parse a config.json file.
    Returns dict with relevant hyperparameters.
    """
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    trainer_config = config.get('trainer', {})
    
    return {
        'seed': trainer_config.get('seed'),
        'k': trainer_config.get('k'),
        'l2_w': trainer_config.get('l2_w'),
        'activation_dim': trainer_config.get('activation_dim'),
        'dict_size': trainer_config.get('dict_size'),
        'steps': trainer_config.get('steps'),
        'layer': trainer_config.get('layer'),
        'lm_name': trainer_config.get('lm_name'),
        'submodule_name': trainer_config.get('submodule_name'),
    }


def find_models_with_configs(model_dir):
    """
    Find all ae.pt files and their associated config.json files.
    Returns list of dicts with model path and parsed config.
    """
    models = []
    
    # Find all config.json files
    config_files = glob.glob(os.path.join(model_dir, '**', 'config.json'), recursive=True)
    
    for config_path in config_files:
        trainer_dir = os.path.dirname(config_path)
        model_path = os.path.join(trainer_dir, 'ae.pt')
        
        if os.path.exists(model_path):
            config = load_config(config_path)
            config['model_path'] = model_path
            config['config_path'] = config_path
            models.append(config)
    
    return models


def group_models_by_config(model_dir):
    """
    Group models by their configuration (k, l2_w), excluding seed.
    Returns a dict: config_key -> list of (seed, model_path, full_config)
    """
    models = find_models_with_configs(model_dir)
    
    grouped = defaultdict(list)
    
    for model_info in models:
        # Create config key (k and l2_w - the hyperparameters that vary)
        config_key = (
            model_info['k'],
            model_info['l2_w'],
        )
        grouped[config_key].append((model_info['seed'], model_info['model_path'], model_info))
    
    # Sort by seed within each group (handle None seeds by placing them at the end)
    for key in grouped:
        grouped[key].sort(key=lambda x: (x[0] is None, x[0] if x[0] is not None else 0))
    
    return dict(grouped)


# Load and group models
model_dirs = [
    '../data_model_weights/random_seeds_constrained_saes_EleutherAI_pythia-70m-deduped_top_k_l2'
]

all_grouped = {}
for model_dir in model_dirs:
    if os.path.exists(model_dir):
        grouped = group_models_by_config(model_dir)
        all_grouped.update(grouped)

# Display discovered configurations
print("Discovered configurations (with multiple seeds):")
print("=" * 80)
for config, seeds_files in sorted(all_grouped.items()):
    if len(seeds_files) >= 2:  # Only show configs with multiple seeds
        k, l2_w = config
        seeds = [s for s, _, _ in seeds_files]
        print(f"k={k}, l2_w={l2_w}")
        print(f"  Seeds: {seeds}")
        print()


## 2. Feature Consistency Metrics

Following Braun et al. (2024), we use **Mean Max Cosine Similarity** as the primary metric:
- For each latent in SAE1, find the maximum cosine similarity with any latent in SAE2
- Average these maxima to get the overall similarity score

We also implement the more conservative **Shared Features** criterion from the paper:
- Compute Hungarian matching on both **encoder** and **decoder** weights
- A feature is "shared" only if both matchings agree AND similarity > 0.7 in both

Key metrics:
1. **Mean Max Cosine Similarity**: Primary metric - does each feature have a good match somewhere?
2. **Fraction Paired (>0.7)**: What fraction of features have max similarity > 0.7?
3. **Shared Features**: Fraction where encoder AND decoder matchings agree with sim > 0.7


In [ ]:
def get_decoder_features(model):
    """Extract decoder features from a model. Returns shape [dict_size, activation_dim]."""
    # decoder.weight has shape [activation_dim, dict_size], so transpose to get features as rows
    return model.decoder.weight.data.T.cpu().numpy()


def get_encoder_features(model):
    """Extract encoder features from a model. Returns shape [dict_size, activation_dim]."""
    # encoder.weight has shape [dict_size, activation_dim]
    return model.encoder.weight.data.cpu().numpy()


def normalize_features(features):
    """Normalize features to unit norm (per row)."""
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    return features / (norms + 1e-8)


def cosine_similarity_matrix(features_a, features_b):
    """
    Compute pairwise cosine similarity between features.
    features_a: [N, D] - N features of dimension D
    features_b: [M, D] - M features of dimension D
    Returns: [N, M] similarity matrix
    """
    a_norm = normalize_features(features_a)
    b_norm = normalize_features(features_b)
    return a_norm @ b_norm.T


def hungarian_matching(sim_matrix):
    """
    Find optimal 1-to-1 matching using Hungarian algorithm.
    Maximizes sum of similarities (not absolute similarities).
    Returns: row_ind, col_ind, matched_sims
    """
    cost_matrix = -sim_matrix  # minimize negative = maximize positive
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    matched_sims = sim_matrix[row_ind, col_ind]
    return row_ind, col_ind, matched_sims


def compute_feature_consistency_metrics(model_a, model_b, threshold=0.7):
    """
    Compute feature consistency metrics following Braun et al. methodology.
    
    Primary metric: Mean Max Cosine Similarity
    - For each feature in A, find max similarity with any feature in B
    - Average these maxima
    
    Also computes "shared features" using both encoder and decoder matchings.
    """
    # Get both encoder and decoder features
    dec_a = get_decoder_features(model_a)
    dec_b = get_decoder_features(model_b)
    enc_a = get_encoder_features(model_a)
    enc_b = get_encoder_features(model_b)
    
    # Compute similarity matrices (using absolute value to handle sign flips)
    dec_sim = np.abs(cosine_similarity_matrix(dec_a, dec_b))
    enc_sim = np.abs(cosine_similarity_matrix(enc_a, enc_b))
    
    # ==========================================================================
    # PRIMARY METRIC: Mean Max Cosine Similarity
    # ==========================================================================
    # For each feature in A, find max similarity with any feature in B
    max_sim_dec_a = np.max(dec_sim, axis=1)  # [N,] - max sim for each feature in A
    max_sim_dec_b = np.max(dec_sim, axis=0)  # [M,] - max sim for each feature in B
    max_sim_enc_a = np.max(enc_sim, axis=1)
    max_sim_enc_b = np.max(enc_sim, axis=0)
    
    # Mean max cosine similarity (primary metric)
    mean_max_cos_dec = (np.mean(max_sim_dec_a) + np.mean(max_sim_dec_b)) / 2
    mean_max_cos_enc = (np.mean(max_sim_enc_a) + np.mean(max_sim_enc_b)) / 2
    
    # Fraction of features with max_sim > threshold
    frac_paired_dec = np.mean(max_sim_dec_a > threshold)
    frac_paired_enc = np.mean(max_sim_enc_a > threshold)
    
    # ==========================================================================
    # SHARED FEATURES: Hungarian matching on both encoder and decoder
    # ==========================================================================
    # Hungarian matching on decoder
    dec_row, dec_col, dec_matched_sims = hungarian_matching(dec_sim)
    
    # Hungarian matching on encoder
    enc_row, enc_col, enc_matched_sims = hungarian_matching(enc_sim)
    
    # Find shared features: where both matchings agree AND both have sim > threshold
    n_features = len(dec_row)
    shared_mask = np.zeros(n_features, dtype=bool)
    
    for i in range(n_features):
        # Check if decoder and encoder matchings agree
        dec_partner = dec_col[i]
        enc_partner = enc_col[i]
        
        if dec_partner == enc_partner:
            # Check if both similarities are above threshold
            if dec_matched_sims[i] > threshold and enc_matched_sims[i] > threshold:
                shared_mask[i] = True
    
    frac_shared = np.mean(shared_mask)
    
    metrics = {
        # Primary metrics (Mean Max Cosine Similarity)
        'mean_max_cos_dec': mean_max_cos_dec,
        'mean_max_cos_enc': mean_max_cos_enc,
        'mean_max_cos_avg': (mean_max_cos_dec + mean_max_cos_enc) / 2,
        
        # Fraction paired (max sim > threshold)
        'frac_paired_dec': frac_paired_dec,
        'frac_paired_enc': frac_paired_enc,
        
        # Shared features (encoder + decoder agree)
        'frac_shared': frac_shared,
        
        # Distribution of max similarities
        'max_sims_dec': max_sim_dec_a,
        'max_sims_enc': max_sim_enc_a,
        
        # Hungarian matching results (for visualization)
        'hungarian_dec_sims': dec_matched_sims,
        'hungarian_enc_sims': enc_matched_sims,
        'dec_matching': (dec_row, dec_col),
        'enc_matching': (enc_row, enc_col),
        'shared_mask': shared_mask,
        
        # Raw similarity matrices
        'dec_sim_matrix': dec_sim,
        'enc_sim_matrix': enc_sim,
    }
    
    return metrics


def analyze_seed_consistency(model_paths, verbose=True):
    """
    Analyze feature consistency across models trained with different seeds.
    model_paths: list of (seed, path, config) tuples or (seed, path) tuples
    """
    # Load all models using AutoEncoderTopK.from_pretrained
    models = {}
    for item in model_paths:
        if len(item) == 3:
            seed, path, config = item
        else:
            seed, path = item
        if verbose:
            print(f"Loading seed {seed}: {os.path.basename(path)}")
        models[seed] = AutoEncoderTopK.from_pretrained(path, device='cpu')
    
    seeds = list(models.keys())
    
    # Analyze all pairs
    results = {}
    for seed_a, seed_b in combinations(seeds, 2):
        metrics = compute_feature_consistency_metrics(models[seed_a], models[seed_b])
        results[(seed_a, seed_b)] = metrics
        
        if verbose:
            print(f"\nSeed {seed_a} vs Seed {seed_b}:")
            print(f"  Mean Max Cos (decoder): {metrics['mean_max_cos_dec']:.4f}")
            print(f"  Mean Max Cos (encoder): {metrics['mean_max_cos_enc']:.4f}")
            print(f"  Frac paired >0.7 (dec): {metrics['frac_paired_dec']:.2%}")
            print(f"  Frac shared (both):     {metrics['frac_shared']:.2%}")
    
    return results, models


## 3. Visualization Functions


In [ ]:
def plot_max_similarity_distribution(results, config_name=""):
    """Plot distribution of MAX similarities (primary metric) across seed pairs."""
    n_pairs = len(results)
    fig, axes = plt.subplots(2, n_pairs, figsize=(5*n_pairs, 8))
    if n_pairs == 1:
        axes = axes.reshape(2, 1)
    
    for col, ((seed_a, seed_b), metrics) in enumerate(results.items()):
        # Decoder max similarities
        ax = axes[0, col]
        ax.hist(metrics['max_sims_dec'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
        ax.axvline(metrics['mean_max_cos_dec'], color='red', linestyle='--', 
                   label=f'Mean: {metrics["mean_max_cos_dec"]:.3f}')
        ax.axvline(0.7, color='green', linestyle=':', linewidth=2, label='Threshold 0.7')
        ax.set_xlabel('Max Cosine Similarity')
        ax.set_ylabel('Count')
        ax.set_title(f'Decoder - Seed {seed_a} vs {seed_b}')
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1)
        
        # Encoder max similarities
        ax = axes[1, col]
        ax.hist(metrics['max_sims_enc'], bins=50, edgecolor='black', alpha=0.7, color='coral')
        ax.axvline(metrics['mean_max_cos_enc'], color='red', linestyle='--', 
                   label=f'Mean: {metrics["mean_max_cos_enc"]:.3f}')
        ax.axvline(0.7, color='green', linestyle=':', linewidth=2, label='Threshold 0.7')
        ax.set_xlabel('Max Cosine Similarity')
        ax.set_ylabel('Count')
        ax.set_title(f'Encoder - Seed {seed_a} vs {seed_b}')
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1)
    
    plt.suptitle(f'Max Cosine Similarity Distribution\n{config_name}', y=1.02)
    plt.tight_layout()
    return fig


def plot_similarity_heatmap(sim_matrix, seed_a, seed_b, title_suffix="", max_features=100):
    """Plot heatmap of similarity matrix (subsampled if too large)."""
    n, m = sim_matrix.shape
    
    # Subsample if too large
    if n > max_features or m > max_features:
        step_n = max(1, n // max_features)
        step_m = max(1, m // max_features)
        sim_sub = sim_matrix[::step_n, ::step_m]
    else:
        sim_sub = sim_matrix
    
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(sim_sub, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='|Cosine Similarity|')
    ax.set_xlabel(f'Features (Seed {seed_b})')
    ax.set_ylabel(f'Features (Seed {seed_a})')
    ax.set_title(f'Feature Similarity Matrix {title_suffix}\n(Seed {seed_a} vs {seed_b})')
    return fig


def plot_matched_features(models, metrics, seed_a, seed_b, n_examples=10, show_shared_only=False):
    """Visualize pairs of matched features side-by-side."""
    features_a = get_decoder_features(models[seed_a])
    features_b = get_decoder_features(models[seed_b])
    
    row_ind, col_ind = metrics['dec_matching']
    max_sims = metrics['max_sims_dec']
    shared_mask = metrics['shared_mask']
    
    if show_shared_only:
        # Only show shared features
        indices = np.where(shared_mask)[0]
        title_suffix = "(Shared Features Only)"
    else:
        # Sort by max similarity to show best matches first
        indices = np.argsort(max_sims)[::-1]
        title_suffix = "(Best Matches)"
    
    n_show = min(n_examples, len(indices))
    fig, axes = plt.subplots(2, n_show, figsize=(2*n_show, 5))
    
    for i in range(n_show):
        idx = indices[i]
        idx_a = row_ind[idx]
        idx_b = col_ind[idx]
        sim = max_sims[idx]
        is_shared = shared_mask[idx]
        
        feat_a = features_a[idx_a].reshape(28, 28)
        feat_b = features_b[idx_b].reshape(28, 28)
        
        # Plot feature from seed A
        ax = axes[0, i]
        vmax = max(np.abs(feat_a).max(), 1e-6)
        ax.imshow(feat_a, cmap='bwr', vmin=-vmax, vmax=vmax)
        shared_str = "✓" if is_shared else ""
        ax.set_title(f'Seed {seed_a} #{idx_a}\nsim={sim:.2f} {shared_str}', fontsize=8)
        ax.axis('off')
        
        # Plot matched feature from seed B
        ax = axes[1, i]
        vmax = max(np.abs(feat_b).max(), 1e-6)
        ax.imshow(feat_b, cmap='bwr', vmin=-vmax, vmax=vmax)
        ax.set_title(f'Seed {seed_b} #{idx_b}', fontsize=8)
        ax.axis('off')
    
    plt.suptitle(f'Matched Features {title_suffix}\nSeed {seed_a} vs {seed_b}', y=1.02)
    plt.tight_layout()
    return fig


def plot_feature_grid(model, n_features=50, title=""):
    """Plot grid of decoder features as images."""
    features = get_decoder_features(model)
    
    n_cols = 10
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.5))
    
    for i in range(n_features):
        row = i // n_cols
        col = i % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        
        feat = features[i].reshape(28, 28)
        vmax = max(np.abs(feat).max(), 1e-6)
        ax.imshow(feat, cmap='bwr', vmin=-vmax, vmax=vmax)
        ax.axis('off')
    
    # Hide empty subplots
    for i in range(n_features, n_rows * n_cols):
        row = i // n_cols
        col = i % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        ax.axis('off')
    
    plt.suptitle(title, y=1.01)
    plt.tight_layout()
    return fig


## 4. Run Analysis Across All Configurations


In [ ]:
# Run analysis on all configurations with multiple seeds
all_analysis_results = {}

for config, seed_files in sorted(all_grouped.items()):
    if len(seed_files) < 2:
        continue
    
    k, l2_w = config
    config_name = f"k={k}, l2_w={l2_w}"
    
    print("=" * 80)
    print(f"Analyzing: {config_name}")
    print("=" * 80)
    
    results, models = analyze_seed_consistency(seed_files, verbose=True)
    all_analysis_results[config] = {'results': results, 'models': models, 'name': config_name}
    print()


## 5. Summary Table


In [ ]:
# Create summary table with new metrics
summary_data = []

for config, data in all_analysis_results.items():
    k, l2_w = config
    
    # Average metrics across all seed pairs
    all_mean_max_cos = []
    all_frac_paired = []
    all_frac_shared = []
    
    for (seed_a, seed_b), metrics in data['results'].items():
        all_mean_max_cos.append(metrics['mean_max_cos_avg'])
        all_frac_paired.append(metrics['frac_paired_dec'])
        all_frac_shared.append(metrics['frac_shared'])
    
    summary_data.append({
        'k': k,
        'l2_w': l2_w,
        'Mean Max Cos': np.mean(all_mean_max_cos),
        'Std': np.std(all_mean_max_cos),
        'Frac Paired (>0.7)': np.mean(all_frac_paired),
        'Frac Shared': np.mean(all_frac_shared),
    })

# Display as formatted table
import pandas as pd
df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values('Mean Max Cos', ascending=False)
print("\nSummary of Feature Consistency Across Seeds:")
print("=" * 100)
print("Mean Max Cos = Mean Max Cosine Similarity (primary metric)")
print("Frac Paired = fraction of features with max_sim > 0.7")
print("Frac Shared = fraction where encoder & decoder matchings agree with sim > 0.7")
print()
display(df_summary.style.format({
    'Mean Max Cos': '{:.4f}',
    'Std': '{:.4f}',
    'Frac Paired (>0.7)': '{:.2%}',
    'Frac Shared': '{:.2%}',
}).background_gradient(subset=['Mean Max Cos', 'Frac Paired (>0.7)'], cmap='RdYlGn'))


## 8. Comparison: Impact of Regularization on Feature Consistency


In [ ]:
# Compare consistency across different regularization settings
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Prepare data for plotting - use all configs for TopK models
configs_all = [(c, d) for c, d in all_analysis_results.items()]

if configs_all:
    labels = []
    mean_max_cos = []
    frac_paired = []
    frac_shared = []
    
    for config, data in sorted(configs_all, key=lambda x: x[0]):
        k, l2_w = config
        label = f"k={k}, l2_w={l2_w}"
        labels.append(label)
        
        # Average across seed pairs
        mean_max_cos.append(np.mean([m['mean_max_cos_avg'] for m in data['results'].values()]))
        frac_paired.append(np.mean([m['frac_paired_dec'] for m in data['results'].values()]))
        frac_shared.append(np.mean([m['frac_shared'] for m in data['results'].values()]))
    
    x = np.arange(len(labels))
    width = 0.6
    
    # Mean Max Cosine Similarity
    bars1 = axes[0].bar(x, mean_max_cos, width, color='steelblue', edgecolor='black')
    axes[0].set_ylabel('Mean Max Cosine Similarity')
    axes[0].set_xlabel('Configuration')
    axes[0].set_title('Primary Metric: Mean Max Cos Sim')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=45, ha='right')
    axes[0].set_ylim(0, 1)
    
    # Fraction Paired (>0.7)
    bars2 = axes[1].bar(x, frac_paired, width, color='coral', edgecolor='black')
    axes[1].set_ylabel('Fraction Paired (>0.7)')
    axes[1].set_xlabel('Configuration')
    axes[1].set_title('Features with Max Sim > 0.7')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=45, ha='right')
    axes[1].set_ylim(0, 1)
    
    # Fraction Shared (conservative)
    bars3 = axes[2].bar(x, frac_shared, width, color='forestgreen', edgecolor='black')
    axes[2].set_ylabel('Fraction Shared')
    axes[2].set_xlabel('Configuration')
    axes[2].set_title('Shared (Enc+Dec agree, >0.7)')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(labels, rotation=45, ha='right')
    axes[2].set_ylim(0, 1)
    
    plt.suptitle('Feature Consistency Comparison', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No configurations found")


## 9. Tied Init vs Non-Tied Init Comparison


## 12. Conclusions

This analysis measures how consistent the learned sparse features are across different random seeds, following methodology from Braun et al. (2024).

**Primary Metric: Mean Max Cosine Similarity**
- For each feature in SAE1, find the maximum cosine similarity with any feature in SAE2
- Average these maxima to get the overall similarity score
- This answers: "Does each feature have a good match somewhere?"

**Secondary Metrics:**
- **Frac Paired (>0.7)**: Fraction of features with max similarity > 0.7
- **Frac Shared**: Conservative metric where encoder AND decoder Hungarian matchings must agree with similarity > 0.7 in both

**Interpretation:**
- High fraction paired (>80%) suggests features are learning true data structure
- Braun et al. found ~42% shared features across independently trained LLM SAEs
- Lower consistency may indicate features are more dependent on initialization


In [ ]:
# Final summary
print("=" * 80)
print("FINAL SUMMARY: Feature Consistency Across Random Seeds")
print("=" * 80)

if all_analysis_results:
    # Find best and worst configurations by Mean Max Cosine Similarity
    best_config = max(all_analysis_results.items(), 
                      key=lambda x: np.mean([m['mean_max_cos_avg'] for m in x[1]['results'].values()]))
    worst_config = min(all_analysis_results.items(),
                       key=lambda x: np.mean([m['mean_max_cos_avg'] for m in x[1]['results'].values()]))
    
    best_mean_max = np.mean([m['mean_max_cos_avg'] for m in best_config[1]['results'].values()])
    best_frac_paired = np.mean([m['frac_paired_dec'] for m in best_config[1]['results'].values()])
    best_frac_shared = np.mean([m['frac_shared'] for m in best_config[1]['results'].values()])
    
    worst_mean_max = np.mean([m['mean_max_cos_avg'] for m in worst_config[1]['results'].values()])
    worst_frac_paired = np.mean([m['frac_paired_dec'] for m in worst_config[1]['results'].values()])
    
    print(f"\n✓ Most consistent features:")
    print(f"  {best_config[1]['name']}")
    print(f"  Mean Max Cos Sim: {best_mean_max:.4f}")
    print(f"  Frac Paired (>0.7): {best_frac_paired:.2%}")
    print(f"  Frac Shared: {best_frac_shared:.2%}")
    
    print(f"\n✗ Least consistent features:")
    print(f"  {worst_config[1]['name']}")
    print(f"  Mean Max Cos Sim: {worst_mean_max:.4f}")
    print(f"  Frac Paired (>0.7): {worst_frac_paired:.2%}")
    
    print("\n" + "=" * 80)
    print("Interpretation (using 0.7 threshold from Braun et al.):")
    print("-" * 80)
    if best_frac_paired > 0.8:
        print("• High consistency: >80% of features have a good match (max sim > 0.7)")
        print("• Features are likely capturing true structure in the data")
    elif best_frac_paired > 0.5:
        print("• Moderate consistency: 50-80% of features have a good match")
        print("• Some features are stable, others are seed-dependent")
    else:
        print("• Low consistency: <50% of features have a good match")
        print("• Features are highly dependent on initialization")
    
    print(f"\n• Shared features (enc+dec agree): {best_frac_shared:.1%} of latents")
    print("  (This is comparable to ~42% found in Braun et al. for LLM SAEs)")
else:
    print("No analysis results available")


## 13. Dead Feature Analysis: Encoder-Decoder Alignment

Dead or poorly trained features might have arbitrary weights that don't contribute to reconstruction. We can identify "alive" features by measuring the cosine similarity between each feature's encoder and decoder weights within the same model.

**Intuition**: For a well-trained feature:
- The encoder weight detects the feature pattern in inputs
- The decoder weight reconstructs that same pattern
- These should be well-aligned (high cosine similarity)

Dead features may have diverged encoder/decoder weights since they're never activated during training.


In [ ]:
def compute_encoder_decoder_alignment(model):
    """
    Compute cosine similarity between encoder and decoder weights for each feature.
    
    For feature i:
    - Encoder weight: encoder.weight[i, :] (how it detects the feature)
    - Decoder weight: decoder.weight[:, i] (how it reconstructs the feature)
    
    Returns array of shape [hidden_dim] with cosine similarities.
    """
    enc = model.encoder.weight.data.cpu().numpy()  # [hidden_dim, input_dim]
    dec = model.decoder.weight.data.cpu().numpy()  # [input_dim, hidden_dim]
    
    # Normalize each feature's weights
    enc_norm = enc / (np.linalg.norm(enc, axis=1, keepdims=True) + 1e-8)
    dec_norm = dec / (np.linalg.norm(dec, axis=0, keepdims=True) + 1e-8)  # normalize columns
    
    # Compute cosine similarity for each feature (dot product of normalized vectors)
    # enc_norm[i, :] dot dec_norm[:, i] = sum(enc_norm[i, :] * dec_norm[:, i])
    alignment = np.sum(enc_norm * dec_norm.T, axis=1)  # [hidden_dim]
    
    return alignment


def get_alive_feature_mask(model, threshold=0.5):
    """
    Get mask of 'alive' features based on encoder-decoder alignment.
    Features with alignment > threshold are considered alive.
    """
    alignment = compute_encoder_decoder_alignment(model)
    return alignment > threshold, alignment


# Compute encoder-decoder alignment for all models
print("Encoder-Decoder Alignment Analysis")
print("=" * 80)

alignment_stats = []

for config, data in all_analysis_results.items():
    k, l2_w = config
    
    models = data['models']
    config_name = data['name']
    
    for seed, model in models.items():
        alignment = compute_encoder_decoder_alignment(model)
        
        stats = {
            'config': config_name,
            'seed': seed,
            'mean_alignment': np.mean(alignment),
            'median_alignment': np.median(alignment),
            'frac_above_0.5': np.mean(alignment > 0.5),
            'frac_above_0.7': np.mean(alignment > 0.7),
            'frac_above_0.9': np.mean(alignment > 0.9),
            'alignment': alignment
        }
        alignment_stats.append(stats)
        
print(f"Analyzed {len(alignment_stats)} models")
print()


## 14. Re-computing Cross-Seed Consistency with Alive Features Only

Now we re-run the consistency analysis but only considering features that are "alive" in both models being compared. This should give us a cleaner picture of whether the *meaningful* features are consistent across seeds.


In [ ]:
def compute_filtered_consistency(model_a, model_b, alignment_threshold=0.2, similarity_threshold=0.7):
    """
    Compute feature consistency only for features that are 'alive' in BOTH models.
    Uses the SAME methodology as compute_feature_consistency_metrics (Braun et al.).
    
    A feature is 'alive' if its encoder-decoder alignment > alignment_threshold.
    Default threshold is 0.2 to only filter truly dead/divergent features.
    """
    # Get alignment for both models
    align_a = compute_encoder_decoder_alignment(model_a)
    align_b = compute_encoder_decoder_alignment(model_b)
    
    alive_mask_a = align_a > alignment_threshold
    alive_mask_b = align_b > alignment_threshold
    
    # Get features
    dec_a = get_decoder_features(model_a)
    dec_b = get_decoder_features(model_b)
    enc_a = get_encoder_features(model_a)
    enc_b = get_encoder_features(model_b)
    
    # Filter to alive features only
    dec_a_alive = dec_a[alive_mask_a]
    dec_b_alive = dec_b[alive_mask_b]
    enc_a_alive = enc_a[alive_mask_a]
    enc_b_alive = enc_b[alive_mask_b]
    
    n_alive_a = int(alive_mask_a.sum())
    n_alive_b = int(alive_mask_b.sum())
    n_total = len(align_a)
    
    if n_alive_a == 0 or n_alive_b == 0:
        return {
            'n_alive_a': n_alive_a,
            'n_alive_b': n_alive_b,
            'n_total': n_total,
            'frac_alive_a': 0.0,
            'frac_alive_b': 0.0,
            'mean_max_cos_dec': np.nan,
            'mean_max_cos_enc': np.nan,
            'mean_max_cos_avg': np.nan,
            'frac_paired_dec': np.nan,
            'frac_paired_enc': np.nan,
            'frac_shared': np.nan,
            'max_sims_dec': np.array([]),
            'max_sims_enc': np.array([]),
        }
    
    # Compute similarity matrices for alive features only (using abs for sign flips)
    dec_sim = np.abs(cosine_similarity_matrix(dec_a_alive, dec_b_alive))
    enc_sim = np.abs(cosine_similarity_matrix(enc_a_alive, enc_b_alive))
    
    # ==========================================================================
    # PRIMARY METRIC: Mean Max Cosine Similarity (same as original)
    # ==========================================================================
    max_sim_dec_a = np.max(dec_sim, axis=1)
    max_sim_dec_b = np.max(dec_sim, axis=0)
    max_sim_enc_a = np.max(enc_sim, axis=1)
    max_sim_enc_b = np.max(enc_sim, axis=0)
    
    mean_max_cos_dec = (np.mean(max_sim_dec_a) + np.mean(max_sim_dec_b)) / 2
    mean_max_cos_enc = (np.mean(max_sim_enc_a) + np.mean(max_sim_enc_b)) / 2
    
    # Fraction paired
    frac_paired_dec = np.mean(max_sim_dec_a > similarity_threshold)
    frac_paired_enc = np.mean(max_sim_enc_a > similarity_threshold)
    
    # ==========================================================================
    # SHARED FEATURES: Hungarian matching on both encoder and decoder (same as original)
    # ==========================================================================
    dec_row, dec_col, dec_matched_sims = hungarian_matching(dec_sim)
    enc_row, enc_col, enc_matched_sims = hungarian_matching(enc_sim)
    
    # Find shared features: where both matchings agree AND both have sim > threshold
    n_features_to_match = min(n_alive_a, n_alive_b)
    shared_mask = np.zeros(n_features_to_match, dtype=bool)
    
    for i in range(n_features_to_match):
        dec_partner = dec_col[i]
        enc_partner = enc_col[i]
        
        if dec_partner == enc_partner:
            if dec_matched_sims[i] > similarity_threshold and enc_matched_sims[i] > similarity_threshold:
                shared_mask[i] = True
    
    frac_shared = np.mean(shared_mask) if len(shared_mask) > 0 else 0.0
    
    return {
        'n_alive_a': n_alive_a,
        'n_alive_b': n_alive_b,
        'n_total': n_total,
        'frac_alive_a': n_alive_a / n_total,
        'frac_alive_b': n_alive_b / n_total,
        # Primary metrics
        'mean_max_cos_dec': mean_max_cos_dec,
        'mean_max_cos_enc': mean_max_cos_enc,
        'mean_max_cos_avg': (mean_max_cos_dec + mean_max_cos_enc) / 2,
        # Fraction paired
        'frac_paired_dec': frac_paired_dec,
        'frac_paired_enc': frac_paired_enc,
        # Shared features (same methodology as original)
        'frac_shared': frac_shared,
        # Distributions
        'max_sims_dec': max_sim_dec_a,
        'max_sims_enc': max_sim_enc_a,
        'hungarian_dec_sims': dec_matched_sims,
        'hungarian_enc_sims': enc_matched_sims,
        'shared_mask': shared_mask,
    }


# Re-run analysis with alive feature filtering
ALIGNMENT_THRESHOLD = 0.1  # Only filter truly dead features (very low enc-dec alignment)

print("Cross-Seed Consistency Analysis: ALIVE FEATURES ONLY")
print("=" * 80)
print(f"(Filtering to features with encoder-decoder alignment > {ALIGNMENT_THRESHOLD})")
print()

filtered_results = {}

for config, data in all_analysis_results.items():
    k, l2_w = config
    
    models = data['models']
    config_name = data['name']
    seeds = list(models.keys())
    
    config_filtered = {}
    
    for seed_a, seed_b in combinations(seeds, 2):
        metrics = compute_filtered_consistency(models[seed_a], models[seed_b], 
                                                alignment_threshold=ALIGNMENT_THRESHOLD)
        config_filtered[(seed_a, seed_b)] = metrics
        
        print(f"{config_name} - Seed {seed_a} vs {seed_b}:")
        print(f"  Alive features: {metrics['n_alive_a']}/{metrics['n_total']} vs {metrics['n_alive_b']}/{metrics['n_total']}")
        print(f"  Mean Max Cos (alive only): {metrics['mean_max_cos_avg']:.4f}")
        print(f"  Frac Paired >0.7 (alive):  {metrics['frac_paired_dec']:.2%}")
        print(f"  Frac Shared (alive):       {metrics['frac_shared']:.2%}")
        print()
    
    filtered_results[config] = {'results': config_filtered, 'name': config_name}


In [ ]:
# Compare filtered vs unfiltered results
print("\nComparison: All Features vs Alive Features Only")
print("=" * 80)
print(f"(Alive = encoder-decoder alignment > {ALIGNMENT_THRESHOLD})")
print()

comparison_data = []

for config in filtered_results.keys():
    if config not in all_analysis_results:
        continue
    
    k, l2_w = config
    config_name = filtered_results[config]['name']
    
    # Average over seed pairs - ALL features
    all_mean_max = np.mean([m['mean_max_cos_avg'] for m in all_analysis_results[config]['results'].values()])
    all_frac_paired = np.mean([m['frac_paired_dec'] for m in all_analysis_results[config]['results'].values()])
    all_frac_shared = np.mean([m['frac_shared'] for m in all_analysis_results[config]['results'].values()])
    
    # Average over seed pairs - ALIVE features only
    filtered_metrics = [m for m in filtered_results[config]['results'].values() if not np.isnan(m['mean_max_cos_avg'])]
    filtered_mean_max = np.mean([m['mean_max_cos_avg'] for m in filtered_metrics])
    filtered_frac_paired = np.mean([m['frac_paired_dec'] for m in filtered_metrics])
    filtered_frac_shared = np.mean([m['frac_shared'] for m in filtered_metrics])
    
    frac_alive = np.mean([m['frac_alive_a'] for m in filtered_results[config]['results'].values()])
    
    comparison_data.append({
        'Config': config_name,
        'k': k,
        'l2_w': l2_w,
        'Frac Alive': frac_alive,
        'All: Mean Max Cos': all_mean_max,
        'Alive: Mean Max Cos': filtered_mean_max,
        'Δ Mean Max Cos': filtered_mean_max - all_mean_max,
        'All: Frac Shared': all_frac_shared,
        'Alive: Frac Shared': filtered_frac_shared,
        'Δ Frac Shared': filtered_frac_shared - all_frac_shared,
    })

df_comparison = pd.DataFrame(comparison_data)
print("\nImpact of Dead Feature Filtering (using same methodology as Braun et al.):")
display(df_comparison.style.format({
    'Frac Alive': '{:.1%}',
    'All: Mean Max Cos': '{:.4f}',
    'Alive: Mean Max Cos': '{:.4f}',
    'Δ Mean Max Cos': '{:+.4f}',
    'All: Frac Shared': '{:.1%}',
    'Alive: Frac Shared': '{:.1%}',
    'Δ Frac Shared': '{:+.1%}',
}).background_gradient(subset=['Δ Mean Max Cos', 'Δ Frac Shared'], cmap='RdYlGn'))


In [ ]:
import matplotlib.pyplot as plt

# Highlight regularized (l2_w > 0) vs non-regularized (l2_w = 0)
fig, axes = plt.subplots(1, 3, figsize=(23, 7), constrained_layout=True)

# Prepare data for plotting - get configs present in both datasets
configs_all = {c: d for c, d in all_analysis_results.items()}
configs_filtered = {c: d for c, d in filtered_results.items()}
common_configs = sorted(set(configs_all.keys()) & set(configs_filtered.keys()))

if common_configs:
    # We'll separate "above label" text and main label text
    main_labels = []
    above_labels = []  # Will be REG or NO REG
    all_mean_max_cos = []
    all_frac_paired = []
    all_frac_shared = []
    filtered_mean_max_cos = []
    filtered_frac_paired = []
    filtered_frac_shared = []
    reg_status = []

    for config in common_configs:
        k, l2_w = config
        main_label = f"k={k}"
        if l2_w == 0:
            above_label = "NO REG"
            reg_status.append('nonreg')
        else:
            above_label = "REG"
            reg_status.append('reg')
        above_labels.append(above_label)
        main_labels.append(main_label)

        all_data = configs_all[config]
        all_mean_max_cos.append(np.mean([m['mean_max_cos_avg'] for m in all_data['results'].values()]))
        all_frac_paired.append(np.mean([m['frac_paired_dec'] for m in all_data['results'].values()]))
        all_frac_shared.append(np.mean([m['frac_shared'] for m in all_data['results'].values()]))

        filtered_data = configs_filtered[config]
        valid_metrics = [m for m in filtered_data['results'].values() if not np.isnan(m['mean_max_cos_avg'])]
        filtered_mean_max_cos.append(np.mean([m['mean_max_cos_avg'] for m in valid_metrics]) if valid_metrics else 0)
        filtered_frac_paired.append(np.mean([m['frac_paired_dec'] for m in valid_metrics]) if valid_metrics else 0)
        filtered_frac_shared.append(np.mean([m['frac_shared'] for m in valid_metrics]) if valid_metrics else 0)

    x = np.arange(len(main_labels))
    width = 0.35

    label_fontsize = 20
    tick_fontsize = 20
    legend_fontsize = 20
    title_fontsize = 22
    suptitle_fontsize = 25

    # More vertical spacing for above-label annotation
    bottom_room = 0.30
    above_label_room = 0.05

    reg_bar_kwargs = dict(edgecolor='red', linewidth=3, hatch='////')
    nonreg_bar_kwargs = dict(edgecolor='black')

    def get_bar_kwargs(idx):
        return reg_bar_kwargs if reg_status[idx] == 'reg' else nonreg_bar_kwargs

    from matplotlib.patches import Patch

    # We'll store legend handles during plotting on the first subplot
    legend_handles = None

    # Main plotting loop
    for idx in range(len(x)):
        # For the bars, keep red outline for REG, normal for NO REG
        bar_kwargs = get_bar_kwargs(idx)
        if idx == 0:
            b1 = axes[0].bar(x[idx] - width/2, all_mean_max_cos[idx], width,
                        color='steelblue', alpha=0.8, edgecolor='black',
                        label='All Features')
            b2 = axes[0].bar(x[idx] + width/2, filtered_mean_max_cos[idx], width,
                        color='darkorange', alpha=0.85, edgecolor='black',
                        label=f'Alive Only (align>{ALIGNMENT_THRESHOLD})')
        else:
            b1 = axes[0].bar(x[idx] - width/2, all_mean_max_cos[idx], width,
                        color='steelblue', alpha=0.8, **bar_kwargs,
                        label=None)
            b2 = axes[0].bar(x[idx] + width/2, filtered_mean_max_cos[idx], width,
                        color='darkorange', alpha=0.85, **bar_kwargs,
                        label=None)
    axes[0].set_ylabel('Mean Max Cosine Similarity', fontsize=label_fontsize)
    # axes[0].set_xlabel('Configuration', fontsize=label_fontsize)  # Removed as per instructions
    axes[0].set_title('Primary Metric: Mean Max Cos Sim', fontsize=title_fontsize, pad=18)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(['']*len(x))
    axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis='y', labelsize=tick_fontsize)
    axes[0].margins(y=0.12)
    # Legend: manually specify patches so only 'All Features' and 'Alive Only' are simple bars,
    # and REG is visually patched separately
    legend_handles = [
        Patch(facecolor='steelblue', edgecolor='black', label='All Features'),
        Patch(facecolor='darkorange', edgecolor='black', label=f'Alive Only (align>{ALIGNMENT_THRESHOLD})'),
        Patch(facecolor='white', edgecolor='red', hatch='////', linewidth=3, label='REG (l2_w>0)')
    ]

    for idx in range(len(x)):
        bar_kwargs = get_bar_kwargs(idx)
        axes[1].bar(x[idx] - width/2, all_frac_paired[idx], width,
                    color='steelblue', alpha=0.8, **bar_kwargs,
                    label=None)
        axes[1].bar(x[idx] + width/2, filtered_frac_paired[idx], width,
                    color='darkorange', alpha=0.85, **bar_kwargs,
                    label=None)
    axes[1].set_ylabel('Fraction Paired (>0.7)', fontsize=label_fontsize)
    # axes[1].set_xlabel('Configuration', fontsize=label_fontsize)  # Removed as per instructions
    axes[1].set_title('Features with Max Sim > 0.7', fontsize=title_fontsize, pad=18)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(['']*len(x))
    axes[1].set_ylim(0, 1)
    axes[1].tick_params(axis='y', labelsize=tick_fontsize)
    axes[1].margins(y=0.12)
    # No legend here

    for idx in range(len(x)):
        bar_kwargs = get_bar_kwargs(idx)
        axes[2].bar(x[idx] - width/2, all_frac_shared[idx], width,
                    color='steelblue', alpha=0.8, **bar_kwargs,
                    label=None)
        axes[2].bar(x[idx] + width/2, filtered_frac_shared[idx], width,
                    color='darkorange', alpha=0.85, **bar_kwargs,
                    label=None)
    axes[2].set_ylabel('Fraction Shared', fontsize=label_fontsize)
    # axes[2].set_xlabel('Configuration', fontsize=label_fontsize)  # Removed as per instructions
    axes[2].set_title('Shared (Enc+Dec agree, >0.7)', fontsize=title_fontsize, pad=18)
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(['']*len(x))
    axes[2].set_ylim(0, 1)
    axes[2].tick_params(axis='y', labelsize=tick_fontsize)
    axes[2].margins(y=0.12)
    # No legend here

    # Add above-column text (REG/NO REG) and main label below tick
    for idx in range(len(x)):
        for ax in axes:
            above_label_color = 'red' if reg_status[idx]=='reg' else 'black'
            above_label_weight = 'bold' if reg_status[idx]=='reg' else 'medium'
            ax.text(
                x[idx],               # x position (center)
                -0.07,                # y: above the main label
                above_labels[idx], 
                ha='center',
                va='bottom',
                fontsize=label_fontsize-3,
                color=above_label_color,
                fontweight=above_label_weight,
                transform=ax.get_xaxis_transform()
            )
            # The main label, as usual
            ax.text(
                x[idx], 
                -0.17,                # y: under the above-label
                main_labels[idx], 
                ha='center',
                va='bottom',
                fontsize=label_fontsize,
                rotation=27,
                fontweight='normal',
                linespacing=2,
                transform=ax.get_xaxis_transform()
            )

    # Add grid for easier value reading
    for ax in axes:
        ax.grid(True, axis='y', alpha=0.14, linestyle='--')

    plt.subplots_adjust(bottom=bottom_room + above_label_room)

    # Only 1 legend, in the first subplot, spanning above all subplots if possible
    axes[0].legend(handles=legend_handles, fontsize=legend_fontsize, loc='upper left', bbox_to_anchor=(0,1.08))

    plt.suptitle(
        f'Feature Consistency Comparison \n'
      ,
        y=1.12, fontsize=suptitle_fontsize
    )
    plt.show()
else:
    print("No common configurations found in both datasets")
